# Fairness with respect to hidden attributes (U)

Computes SPD / DIR / EOD grouped by **U** instead of **S**, using the per-sample
`*_predictions_detail.csv` files each model notebook now writes. Since `U` is
continuous (and `n_U` varies across datasets), individuals are grouped via **k-means
(k=2)** on the `U` columns, fit separately per dataset — this generalizes cleanly to
any number of `U` columns, unlike per-column binning.

Output: one `{Model}_{regime}_U_fairness.csv` per model per regime in `model_results/`,
in the same shape as the existing `*_results.csv` files, so `model_benchmark.ipynb` can
load and join them the same way.


In [13]:
# ==========================================
# LIBRARIES
# ==========================================
import os
import glob
import warnings

import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# Shared paths (relative to this notebook's location: Fairness_models/)
PROJECT_ROOT = os.path.abspath("..")
DETAIL_DIR = os.path.join(PROJECT_ROOT, "model_results")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "model_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS = ["CFFair", "CLAIRE", "SRCVAE", "XGBoost", "FairPFN"]

# Non-U columns written by save_predictions_detail() -- everything else in the file
# is a U column, however many there are.
FIXED_COLS = {"model_name", "name_dataset", "row_id", "prediction", "prob_pred", "Y_true", "S0"}
# (unchanged — U_slot_0, U_slot_1, ... are correctly picked up automatically since
# they're not in this set, same as before)

K_CLUSTERS = 2  # matches the binary group structure used for S-based fairness metrics


In [5]:
import os
import glob
import csv
import numpy as np
import pandas as pd

def repair_predictions_detail(broken_path, fixed_path):
    """
    Repairs a *_predictions_detail.csv with ragged column counts. The first 7 fields
    on every row are ALWAYS model_name, name_dataset, row_id, prediction, prob_pred,
    Y_true, S0 -- that part of the schema never changed. Anything after position 7
    is that row's U values, however many there are for that particular dataset.
    No data is lost; only the column layout gets normalized.
    """
    FIXED = ["model_name", "name_dataset", "row_id", "prediction", "prob_pred", "Y_true", "S0"]
    rows = []
    max_u = 0

    with open(broken_path, newline='') as f:
        reader = csv.reader(f)
        next(reader)  # discard the original (unreliable) header
        for line in reader:
            if not line:
                continue
            fixed_part = line[:7]
            u_part = line[7:]
            max_u = max(max_u, len(u_part))
            rows.append(fixed_part + u_part)

    u_col_names = [f"U_slot_{i}" for i in range(max_u)]
    all_cols = FIXED + u_col_names

    records = [r + [np.nan] * (len(all_cols) - len(r)) for r in rows]
    df = pd.DataFrame(records, columns=all_cols)

    for c in ["row_id", "prediction", "prob_pred", "Y_true", "S0"] + u_col_names:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    df.to_csv(fixed_path, index=False)
    return df

# Repair every predictions_detail.csv currently in model_results/, in place.
DETAIL_DIR = "../model_results"  # adjust if you run this from a different folder
broken_files = glob.glob(os.path.join(DETAIL_DIR, "*_predictions_detail.csv"))
print(f"Found {len(broken_files)} detail file(s) to check/repair.")

for path in broken_files:
    fname = os.path.basename(path)
    try:
        df = repair_predictions_detail(path, path)  # overwrite in place
        print(f"  [REPAIRED] {fname}: {len(df)} rows, {len(df.columns)} columns "
              f"(max U columns found: {len(df.columns) - 7})")
    except Exception as e:
        print(f"  [ERROR] {fname}: {e}")

Found 10 detail file(s) to check/repair.
  [REPAIRED] CFFair_semi_predictions_detail.csv: 36000 rows, 11 columns (max U columns found: 4)
  [REPAIRED] CFFair_syn_predictions_detail.csv: 36000 rows, 7 columns (max U columns found: 0)
  [REPAIRED] CLAIRE_semi_predictions_detail.csv: 5520000 rows, 11 columns (max U columns found: 4)
  [REPAIRED] CLAIRE_syn_predictions_detail.csv: 1138000 rows, 47 columns (max U columns found: 40)
  [REPAIRED] FairPFN_semi_syn_predictions_detail.csv: 5520000 rows, 11 columns (max U columns found: 4)
  [REPAIRED] FairPFN_syn_predictions_detail.csv: 1120000 rows, 47 columns (max U columns found: 40)
  [REPAIRED] SRCVAE_semi_predictions_detail.csv: 2520000 rows, 11 columns (max U columns found: 4)
  [REPAIRED] SRCVAE_syn_predictions_detail.csv: 1138000 rows, 47 columns (max U columns found: 40)
  [REPAIRED] XGBoost_semi_predictions_detail.csv: 5520000 rows, 11 columns (max U columns found: 4)
  [REPAIRED] XGBoost_syn_predictions_detail.csv: 1138000 rows, 47 c

## U-fairness computation

For each dataset within a detail file: scale the `U` columns, fit k-means (k=2),
then compute the same SPD / DIR / EOD formulas already used for `S`, keyed on the
resulting cluster label instead.

In [14]:
def compute_u_fairness(sub_df, u_cols, k=K_CLUSTERS, random_state=42):
    """
    sub_df: rows for ONE (model, dataset) test set, from a predictions_detail file.
    u_cols: the U column names present in this file.
    Returns a dict of U-based fairness metrics, or None if clustering isn't possible
    (too few samples, or no U columns at all -- e.g. n_U == 0 configurations).
    """
    n = len(sub_df)
    if len(u_cols) == 0 or n < 2 * k:
        return None

    U = sub_df[u_cols].values
    # Guard against constant/degenerate U columns (zero variance) which would make
    # scaling divide-by-zero; drop those columns before clustering.
    stds = U.std(axis=0)
    keep = stds > 1e-12
    if keep.sum() == 0:
        return None
    U = U[:, keep]

    U_scaled = StandardScaler().fit_transform(U)

    try:
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=random_state)
        cluster_labels = kmeans.fit_predict(U_scaled)
    except Exception:
        return None

    # If k-means collapses to a single effective cluster (e.g. all points identical
    # after scaling), there's no meaningful group split to measure fairness across.
    if len(np.unique(cluster_labels)) < 2:
        return None

    predictions = sub_df["prediction"].values
    y_true = sub_df["Y_true"].values

    group_1_mask = (cluster_labels == 1)
    group_0_mask = (cluster_labels == 0)

    rate_1 = np.mean(predictions[group_1_mask]) if group_1_mask.sum() > 0 else 0
    rate_0 = np.mean(predictions[group_0_mask]) if group_0_mask.sum() > 0 else 0

    stat_parity_diff = abs(rate_1 - rate_0)
    disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float("inf")

    def recall_safe(y_t, y_p):
        pos = y_t == 1
        if pos.sum() == 0:
            return 0.0
        return np.mean(y_p[pos] == 1)

    tpr_1 = recall_safe(y_true[group_1_mask], predictions[group_1_mask]) if group_1_mask.sum() > 0 else 0
    tpr_0 = recall_safe(y_true[group_0_mask], predictions[group_0_mask]) if group_0_mask.sum() > 0 else 0
    equal_opp_diff = abs(tpr_1 - tpr_0)

    return {
        "n_U_cols_used": int(keep.sum()),
        "cluster_size_0": int(group_0_mask.sum()),
        "cluster_size_1": int(group_1_mask.sum()),
        "Statistical_Parity_Diff_wrt_U": round(stat_parity_diff, 4),
        "Disparate_Impact_Ratio_wrt_U": round(disp_impact, 4),
        "Equal_Opportunity_Diff_wrt_U": round(equal_opp_diff, 4),
        "Pos_Rate_U_cluster1": round(rate_1, 4),
        "Pos_Rate_U_cluster0": round(rate_0, 4),
    }


def process_detail_file(detail_path, model_name, regime):
    """Loads one *_predictions_detail.csv and computes U-fairness per dataset within it."""
    df = pd.read_csv(detail_path)
    u_cols = [c for c in df.columns if c not in FIXED_COLS]

    results = []
    for dataset_name, sub_df in df.groupby("name_dataset"):
        metrics = compute_u_fairness(sub_df, u_cols)
        if metrics is None:
            print(f"  [SKIPPED] {dataset_name}: not enough U columns/samples to cluster.")
            continue

        results.append({
            "model_name": model_name,
            "name_dataset": dataset_name,
            "total_samples": len(sub_df),
            **metrics,
        })

    return pd.DataFrame(results)


## Synthetic (SF) — U-fairness

In [15]:
for model in MODELS:
    detail_path = os.path.join(DETAIL_DIR, f"{model}_syn_predictions_detail.csv")
    output_path = os.path.join(OUTPUT_DIR, f"{model}_syn_U_fairness.csv")

    if not os.path.isfile(detail_path):
        print(f"[{model}] No detail file found at {detail_path} -- skipping.")
        continue

    print(f"[{model}] Processing {detail_path} ...")
    result_df = process_detail_file(detail_path, model, "syn")

    if len(result_df) > 0:
        result_df.to_csv(output_path, index=False)
        print(f"  [SUCCESS] {len(result_df)} dataset(s) processed. Saved to {output_path}")
    else:
        print(f"  [WARNING] No datasets could be processed for {model} (synthetic).")


[CFFair] Processing c:\Users\patri\Documents\Master Semesters\2nd semester\APA\Fairness-evaluation-in-the-presence-of-unobserved-unfair-attributes\model_results\CFFair_syn_predictions_detail.csv ...
  [SKIPPED] SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_0_8_nodes_full_data.csv: not enough U columns/samples to cluster.
  [SKIPPED] SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_1_8_nodes_full_data.csv: not enough U columns/samples to cluster.
  [SKIPPED] SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_2_8_nodes_full_data.csv: not enough U columns/samples to cluster.
  [SKIPPED] SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_3_8_nodes_full_data.csv: not enough U columns/samples to cluster.
  [SKIPPED] SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_4_8_nodes_full_data.csv: not enough U columns/samples to cluster.
  [SKIPPED] SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_b

## Semi-Synthetic (HR) — U-fairness

In [16]:
for model in MODELS:
    detail_path = os.path.join(DETAIL_DIR, f"{model}_semi_predictions_detail.csv")
    output_path = os.path.join(OUTPUT_DIR, f"{model}_semi_syn_U_fairness.csv")

    if not os.path.isfile(detail_path):
        print(f"[{model}] No detail file found at {detail_path} -- skipping.")
        continue

    print(f"[{model}] Processing {detail_path} ...")
    result_df = process_detail_file(detail_path, model, "semi")

    if len(result_df) > 0:
        result_df.to_csv(output_path, index=False)
        print(f"  [SUCCESS] {len(result_df)} dataset(s) processed. Saved to {output_path}")
    else:
        print(f"  [WARNING] No datasets could be processed for {model} (semi-synthetic).")


[CFFair] Processing c:\Users\patri\Documents\Master Semesters\2nd semester\APA\Fairness-evaluation-in-the-presence-of-unobserved-unfair-attributes\model_results\CFFair_semi_predictions_detail.csv ...
  [SKIPPED] HR_N10000_HighBias_Thresh1.0_Seed1000_BIASED_FAIRNESS.csv: not enough U columns/samples to cluster.
  [SKIPPED] HR_N10000_HighBias_Thresh1.0_Seed100_BIASED_FAIRNESS.csv: not enough U columns/samples to cluster.
  [SKIPPED] HR_N10000_HighBias_Thresh1.0_Seed1100_BIASED_FAIRNESS.csv: not enough U columns/samples to cluster.
  [SKIPPED] HR_N10000_HighBias_Thresh1.0_Seed1200_BIASED_FAIRNESS.csv: not enough U columns/samples to cluster.
  [SKIPPED] HR_N10000_HighBias_Thresh1.0_Seed1300_BIASED_FAIRNESS.csv: not enough U columns/samples to cluster.
  [SKIPPED] HR_N10000_HighBias_Thresh1.0_Seed1400_BIASED_FAIRNESS.csv: not enough U columns/samples to cluster.
  [SKIPPED] HR_N10000_HighBias_Thresh1.0_Seed1500_BIASED_FAIRNESS.csv: not enough U columns/samples to cluster.
  [SKIPPED] HR_N1

## Quick sanity check

Loads everything just written and prints summary stats -- a fast way to confirm the
output looks reasonable before switching over to `model_benchmark.ipynb`.

In [17]:
for regime, suffix in [("Synthetic", "_syn_U_fairness.csv"), ("Semi-Synthetic", "_semi_syn_U_fairness.csv")]:
    print(f"=== {regime} ===")
    for model in MODELS:
        path = os.path.join(OUTPUT_DIR, f"{model}{suffix}")
        if os.path.isfile(path):
            d = pd.read_csv(path)
            print(f"  {model}: {len(d)} rows | mean SPD_wrt_U = {d['Statistical_Parity_Diff_wrt_U'].mean():.4f}")
        else:
            print(f"  {model}: (no output file)")
    print()


=== Synthetic ===
  CFFair: 60 rows | mean SPD_wrt_U = 0.5881
  CLAIRE: 60 rows | mean SPD_wrt_U = 0.6403
  SRCVAE: 60 rows | mean SPD_wrt_U = 0.2628
  XGBoost: 60 rows | mean SPD_wrt_U = 0.6296
  FairPFN: 60 rows | mean SPD_wrt_U = 0.4927

=== Semi-Synthetic ===
  CFFair: 800 rows | mean SPD_wrt_U = 0.1406
  CLAIRE: 800 rows | mean SPD_wrt_U = 0.1100
  SRCVAE: 800 rows | mean SPD_wrt_U = 0.0197
  XGBoost: 800 rows | mean SPD_wrt_U = 0.1609
  FairPFN: 800 rows | mean SPD_wrt_U = 0.1194

